# Integrated Gradients — Axiomatic Attribution for Deep Networks (2017)

_Papers · Meta-learning, Bayesian & Robustness_

**Attribute a prediction to its inputs by integrating gradients along a straight path from a baseline to the input.**

---

This notebook is a practice scaffold for the **Integrated Gradients — Axiomatic Attribution for Deep Networks (2017)** lesson. Run the cells, then tackle the practice problems at the bottom. _Save a copy to your Drive (File → Save a copy in Drive) to edit and keep your work._

In [ ]:
# Setup — numpy / pandas / scikit-learn / matplotlib ship with Colab.
import numpy as np, matplotlib.pyplot as plt

## Reference implementation — PyTorch

In [ ]:
import torch, torch.nn as nn
import numpy as np
torch.manual_seed(0); np.random.seed(0)

# === The novel method, built by hand: Integrated Gradients (paper Eqn. 3). ===
def integrated_gradients(F, x, baseline, steps=50):
    """Riemann sum of input-gradients along the straight path baseline -> x."""
    total = torch.zeros_like(x)
    for k in range(1, steps + 1):            # alpha = k/steps, sweeps (0, 1]
        alpha = k / steps
        xi = (baseline + alpha * (x - baseline)).clone().requires_grad_(True)
        g, = torch.autograd.grad(F(xi), xi)  # dF/dx at this path point
        total += g
    return (x - baseline) * total / steps    # scale by travel distance (x - x')

# --- Cell 1. Worked example: F(x1,x2) = x1*x2, x=(1,2), baseline=(0,0). ---
xp = torch.tensor([0.0, 0.0]); x = torch.tensor([1.0, 2.0])
def Fmul(z): return z[..., 0] * z[..., 1]
ig = integrated_gradients(Fmul, x, xp, steps=50)
print("worked example  F(x1,x2)=x1*x2,  x=(1,2),  baseline=(0,0)")
print("  IG (50 steps) =", [round(v, 4) for v in ig.tolist()], " sum =", round(ig.sum().item(), 4))
print("  F(x)-F(xp)    =", round((Fmul(x) - Fmul(xp)).item(), 4), " <- completeness target")
xg = x.clone().requires_grad_(True); g, = torch.autograd.grad(Fmul(xg), xg)
plain = g * (x - xp)
print("  plain grad*input =", [round(v, 4) for v in plain.tolist()], " sum =", round(plain.sum().item(), 4),
      " <- overshoots the gap (no completeness)")
# IG (50 steps) = [1.02, 1.02]  sum = 2.04   (exact integral is (1,1), sum 2.0)
# F(x)-F(xp)    = 2.0
# plain grad*input = [2.0, 2.0]  sum = 4.0

# --- Cell 2. A tiny torch.nn model; verify COMPLETENESS on it. ---
torch.manual_seed(1)
net = nn.Sequential(nn.Linear(4, 8), nn.ReLU(), nn.Linear(8, 1))
def Fnet(z): return net(z).squeeze(-1)        # scalar output to attribute
xb  = torch.tensor([0.7, -1.2, 0.3, 2.0])     # input to explain
xpb = torch.zeros(4)                          # baseline x' = zeros

ig_net = integrated_gradients(Fnet, xb, xpb, steps=50)
sum_ig = ig_net.sum().item()
delta  = (Fnet(xb) - Fnet(xpb)).item()
print("\nmulti-layer perceptron (4->8->1, ReLU)")
print("  IG per feature =", [round(v, 4) for v in ig_net.tolist()])
print("  sum IG         = %.5f" % sum_ig)
print("  F(x) - F(xp)   = %.5f" % delta)
print("  COMPLETENESS gap = %.6f  (~0: attributions sum to the prediction gap)" % (sum_ig - delta))

xgb = xb.clone().requires_grad_(True); gb, = torch.autograd.grad(Fnet(xgb), xgb)
plain_sum = (gb * (xb - xpb)).sum().item()
print("  plain grad*input sum = %.5f   gap vs F(x)-F(xp) = %.5f  (no completeness)" %
      (plain_sum, plain_sum - delta))
# IG per feature = [0.0685, 0.0907, -0.0506, 0.0186]
# sum IG         = 0.12723
# F(x) - F(xp)   = 0.12721
# COMPLETENESS gap = 0.000027        <- Integrated Gradients sums to the gap
# plain grad*input sum = 0.15238   gap vs F(x)-F(xp) = 0.02517   <- plain misses it
# (Our small run, not the paper's reported numbers. Values vary by seed/hardware.)

## Visualize the data & results

_As we take more Riemann steps, does the Integrated-Gradients sum converge to the prediction gap F(x) - F(x') — and does plain grad*input ever get there?_

In [ ]:
import torch, torch.nn as nn, numpy as np
torch.manual_seed(1); np.random.seed(0)

# tiny multi-layer perceptron; scalar output to attribute
net = nn.Sequential(nn.Linear(4, 8), nn.ReLU(), nn.Linear(8, 1))
def Fnet(z): return net(z).squeeze(-1)
xb = torch.tensor([0.7, -1.2, 0.3, 2.0]); xpb = torch.zeros(4)
delta = (Fnet(xb) - Fnet(xpb)).item()                 # the gap to match

def ig_sum(steps):
    tot = torch.zeros(4)
    for k in range(1, steps + 1):
        a = k / steps
        xi = (xpb + a * (xb - xpb)).clone().requires_grad_(True)
        g, = torch.autograd.grad(Fnet(xi), xi); tot += g
    return ((xb - xpb) * tot / steps).sum().item()

# plain grad*input sum (no path -> constant in steps)
xg = xb.clone().requires_grad_(True); g, = torch.autograd.grad(Fnet(xg), xg)
plain = (g * (xb - xpb)).sum().item()

steps = [5, 10, 20, 50, 100]
ig_gap    = [round(abs(ig_sum(s) - delta), 5) for s in steps]
plain_gap = [round(abs(plain - delta), 5) for _ in steps]
print("steps          :", steps)
print("IG    |gap|    :", ig_gap)
print("plain |gap|    :", plain_gap)
# steps          : [5, 10, 20, 50, 100]
# IG    |gap|    : [0.00131, 0.0012, 0.0012, 3e-05, 3e-05]
# plain |gap|    : [0.02517, 0.02517, 0.02517, 0.02517, 0.02517]
# IG -> 0 as steps grow; plain stays flat. Our small run, not the paper's number.

## Practice

Try each one in the empty cell below it, then reveal the worked solution.

**Problem 1.** The completeness ablation. You have a working Integrated-Gradients implementation that sums
            (almost) to $F(x) - F(x')$. Replace it with plain grad(F(x), x) * (x - x')
            evaluated only at the input. On the product model $F = x_1 x_2$ with $x = (1,2)$, $x' = (0,0)$, what
            does each method's attribution sum to, and which property did you just break?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Integrated Gradients: $\text{IG} = (1, 1)$, sum $= 2$. — _From the worked example: integrating $\partial F/\partial x_1 = 2\alpha$ and $\partial F/\partial x_2 = \alpha$ along the path, scaled by $(x - x')$, gives $(1, 1)$._
- Plain grad times travel: gradient at $x=(1,2)$ is $(x_2, x_1) = (2, 1)$; times $(x - x') = (1, 2)$ gives $(2, 2)$, sum $= 4$. — _Plain uses the slope only at the input, ignoring how the slope changes along the path, so the interaction is double-counted._
- Compare to the gap $F(x) - F(x') = 2 - 0 = 2$. — _Integrated Gradients matches it ($2 = 2$); plain overshoots ($4 \ne 2$) &mdash; Completeness is broken._

**Answer:** Integrated Gradients sums to $\mathbf{2}$, exactly the prediction gap $F(x) - F(x') = 2$ &mdash;
                 Completeness holds. Plain grad * (x - x') sums to $\mathbf{4}$, double the
                 gap &mdash; Completeness broken. The product model is nonlinear (an interaction), so the
                 slope at the input alone over-credits both features; averaging the slope along the whole path
                 splits the $2$ evenly into $(1, 1)$.

</details>

**Problem 2.** Why can a plain gradient assign zero attribution to a feature that clearly mattered,
            and how does the path integral rescue it? Reason with a saturating model.

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Picture a model whose output rises with a feature but then saturates (flattens) once the feature is large &mdash; the gradient there is $\approx 0$. — _A plain gradient reads the local slope; in the flat region that slope is near zero even though moving the feature from baseline to here changed the output a lot._
- Now walk from baseline $x'$ to input $x$: early on the path the output is still rising, so the gradient is large; only near the input is it flat. — _Integrated Gradients averages the gradient over the whole path, so it captures the steep early part the input-only gradient missed._
- By Completeness the credits still sum to $F(x) - F(x')$, so the important feature gets its due share. — _The fundamental theorem of calculus guarantees the path integral recovers the full output change._

**Answer:** A plain gradient measures the slope only at the input. If the model has saturated there
                 &mdash; gone flat &mdash; that slope is near zero, so the feature looks unimportant even though it
                 drove the prediction up from the baseline. Integrated Gradients integrates the gradient along the
                 whole path from baseline to input, so it sees the steep, non-saturated stretch the input
                 gradient missed. Completeness then forces the credits to sum to the true gap, so the feature gets
                 a non-zero, faithful attribution (this is exactly the Sensitivity axiom, &sect;2.1).

</details>

**Problem 3.** In the worked example a $50$-step Riemann sum gave $\sum_i \text{IG}_i \approx 2.04$, not the exact
            $2.0$. Is the method wrong? What single knob shrinks the gap, and which direction?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Identify the cause: Equation 3 is a finite right-endpoint Riemann sum sampling $\alpha = k/m$. On the increasing integrand it slightly over-estimates. — _A right-endpoint sum on a rising function sits above the true area; the overshoot is an approximation artifact, not a flaw in the method._
- Raise $m$ (the step count): at $m = 5$ the sum is $2.40$, at $m = 50$ it is $2.04$, at $m = 300$ it is $\approx 2.007$. — _More, finer steps make the Riemann sum approach the exact integral, so the completeness gap shrinks toward zero._
- Recall the paper's guidance: "20 to 300 steps are enough ... (within 5%)" (&sect;5). — _It quantifies the trade-off: a few hundred steps gets you within a few percent for typical models._

**Answer:** The method is correct; the $0.04$ is pure discretization error from the finite
                 Riemann sum (right-endpoint, on an increasing integrand, so it overshoots). The single knob is
                 the step count $m$ &mdash; increase it. Our run: $m = 5 \to 2.40$, $m = 50 \to 2.04$,
                 $m = 300 \to 2.007$; the gap shrinks toward zero. The true integral satisfies Completeness
                 exactly; the paper notes $20$ to $300$ steps land within $5\%$ (&sect;5). Our small run, not the
                 paper's number.

</details>